In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os
import requests
import sys
import json
import seaborn as sns
import matplotlib.pyplot as plt
import Bio
from Bio.PDB.PDBParser import PDBParser
from Bio.PDB.Polypeptide import PPBuilder
from Bio import pairwise2
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio import SeqIO
from Bio.SeqUtils.ProtParam import ProteinAnalysis
#from sklearn.preprocessing import LabelEncoder, OneHotEncoder  
import gc
#from scipy import sparse
import gc
from tqdm import tqdm
#import torch
#import torch.nn as nn
#import torch.nn.functional as TF
#from torch.utils.data import Dataset, DataLoader



In [ ]:
df_train = pd.read_csv('/kaggle/input/novozymes-enzyme-stability-prediction/train.csv', index_col=0).drop(['data_source'], axis=1).dropna(how='all')
upd=pd.read_csv('/kaggle/input/novozymes-enzyme-stability-prediction/train_updates_20220929.csv', index_col=0).drop(['data_source'], axis=1).dropna(how='all')
df_train.loc[upd.index]=upd
print(df_train.shape)
df_train.head()

In [ ]:
wildtype='VPVNPEPDATSVENVALKTGSGDSQSDPIKADLEVKGQSALPFDVDCWAILCKGAPNVLQRVNEKTKNSNRDRSGANKGPFKDPQKWGIKALPPKNPSWSAQDFKSPEEYAFASSLQGGTNAILAPVNLASQNSQGGVLNGFYSANKVAQFDPSKPQQTKGTWFQITKFTGAAGPYCKALGSNDKSVCDKNKNIAGDWGFDPAKWAYQYDEKNNKFNYVGK'

In [ ]:
parser=PDBParser(QUIET=True)
wild_structure=parser.get_structure('wspa','/kaggle/input/novozymes-enzyme-stability-prediction/wildtype_structure_prediction_af2.pdb')
ppb=PPBuilder()
seq=ppb.build_peptides(wild_structure)[0].get_sequence()
assert seq == wildtype

In [ ]:
df_test = pd.read_csv('/kaggle/input/novozymes-enzyme-stability-prediction/test.csv')
df_test.head()

In [ ]:
records=pd.Series(index=df_test.index, dtype=str)
position=pd.Series(index=df_test.index, dtype=int)
wild=pd.Series(index=df_test.index, dtype=str)
subst=pd.Series(index=df_test.index, dtype=str)
for i in df_test.index:
    alignment = pairwise2.align.globalxs(seq, df_test.iloc[i]['protein_sequence'], -1, -1)[0]
    r=''
    ch=1
    for pos in range(len(seq)):
        if alignment.seqA[pos]=='-':
            ch-=1
        if alignment.seqA[pos]!=alignment.seqB[pos]:
            r=' '.join([r,alignment.seqA[pos]+str(pos+ch)+alignment.seqB[pos]])
    r=r.strip()
    records.iloc[i]=r
    if r!='':
        position.iloc[i]=int(r[1:-1])
        wild.iloc[i]=r[0]
        subst.iloc[i]=r[-1]
df_test['mutations']=records
df_test['position']=position
df_test['wild']=wild
df_test['subst']=subst


In [ ]:
df_test.head()

Use GEMME for prediction of stability change

In [ ]:
records=[x for x in records if ('-' not in x) and (x!='')]
len(records)
with open('mutations.txt', 'w') as f:
    f.write('\n'.join(records))

In [ ]:
record = SeqRecord(Seq(wildtype))
SeqIO.write(record, "query.fasta", "fasta")


Use these files to run GEMME (http://www.lcqb.upmc.fr/GEMME/Home.html)

In [ ]:
result_page='http://www.lcqb.upmc.fr/GEMME/report_results.php?jobId=88GbkRwE'

In [ ]:
!wget http://www.lcqb.upmc.fr/GEMME/jobs/88GbkRwE/gemme_88GbkRwE.tar

In [ ]:
import shutil
shutil.unpack_archive('gemme_88GbkRwE.tar')

In [ ]:
arr=[]
with open('unknown/normPred_evolCombi.txt') as f:
    x=f.readline()
    for x in f.readlines():
        mut=x.split()[0].replace('"','')
        d=float(x.split()[1])
        arr.append([mut,d])

In [ ]:
df_subm=pd.DataFrame(columns=['mutations','gemme'], data=arr)
df_test=df_test.merge(df_subm,on='mutations',how='left')
df_test

In [ ]:
df_test.to_csv('gemme.csv', index=False)

In [ ]:
df_subm=df_test[['seq_id','gemme']].fillna(0)
df_subm.columns=['seq_id','tm']
df_subm.to_csv("submission.csv", index = False)

In [ ]:
df_subm